In [ ]:
import pandas as pd
import pandas_ta as ta
initial_df = pd.read_csv("btcusdt_1h.csv")
atr = pd.read_csv("atr.csv")

final_df = initial_df.copy()
initial_df

In [ ]:
from ta.utils import dropna
from ta.volume import OnBalanceVolumeIndicator

# Initialize On Balance Volume Indicator
indicator_obv = OnBalanceVolumeIndicator(close=final_df["close"],volume=final_df["volume"])
final_df['obv_values'] = indicator_obv.on_balance_volume()
#ema = exponential moving average
final_df["obv_ema"] = final_df['obv_values'].ewm(span=200, adjust=False).mean()

In [ ]:
from ta.volatility import BollingerBands

# Initialize Bollinger Bands Indicator
indicator_bb = BollingerBands(close=final_df["close"], window=25, window_dev=2.5)
curr_sig=0
closed=True
final_df['bb_bbm_vals'] = indicator_bb.bollinger_mavg()
final_df['bb_bbh_vals'] = indicator_bb.bollinger_hband()
final_df['bb_bbl_vals'] = indicator_bb.bollinger_lband()

In [ ]:
from ta.utils import dropna
from ta.volume import OnBalanceVolumeIndicator

# Initialize On Balance Volume Indicator
indicator_obv = OnBalanceVolumeIndicator(close=final_df["close"],volume=final_df["volume"])
final_df['obv_values'] = indicator_obv.on_balance_volume()
#ema = exponential moving average
final_df["obv_ema"] = final_df['obv_values'].ewm(span=200, adjust=False).mean()

In [ ]:


#condition to enter long trade
def strat_long_entry(final_df,bar):

    if final_df["ema_25"].iloc[bar]>final_df["ema_50"].iloc[bar] :
        return True
    else:
        return False

# condition to enter short trade
def strat_short_entry(final_df,bar):
    if final_df["ema_25"].iloc[bar]<final_df["ema_50"].iloc[bar] :
        return True
    else:
        return False

#condition to exit long trade
def strat_long_exit(final_df,bar):

    if final_df["ema_25"].iloc[bar]<final_df["ema_70"].iloc[bar] and final_df["ema_25"].iloc[bar-1]>=final_df["ema_70"].iloc[bar-1]:
        return True
    else:
        return False

#condition to exit short trade
def strat_short_exit(final_df,bar):
    if final_df["ema_25"].iloc[bar]>final_df["ema_70"].iloc[bar] and final_df["ema_25"].iloc[bar-1]<=final_df["ema_70"].iloc[bar-1]:
        return True
    else:
        return False

#stop loss conditions for long and short trades
def long_stop_loss(bar,long_entry_price , k):
    if long_entry_price==None:
        return False
    elif final_df["close"].iloc[bar]<long_entry_price-k*final_df["atr"].iloc[bar]\
        and final_df["bb_bbl_vals"].iloc[bar]>final_df["close"].iloc[bar]\
        and final_df["obv_ema"].iloc[bar]>final_df["obv_values"].iloc[bar]:
        return True
    else:
        return False
    

def short_stop_loss(bar,short_entry_price , k):
    
    if short_entry_price==None:
        return False
    elif final_df["close"].iloc[bar]>short_entry_price+k*final_df["atr"].iloc[bar]\
    and final_df["bb_bbh_vals"].iloc[bar]<final_df["close"].iloc[bar]\
    and final_df["obv_values"].iloc[bar]>final_df["obv_ema"].iloc[bar]:
        return True
    else:
        return False

In [ ]:
#import both csvs:- 1. historical data 2. strategy signals
import backtesting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import math
from datetime import datetime

#read csvs
#historical data
hist = pd.read_csv('btcusdt_1h.csv')
strat = pd.read_csv('output_new.csv')

#format of hist
#datetime,open,high,low,close,volume

#format of strat
#datetime,signals,open,high,low,close,volume,


# 1. open long
class backtest():
    def __init__(self, hist, strat):
        self.is_static = True
        self.trade_signal = 0
        self.hist = hist
        self.strat = strat
        self.signals = self.extract_signals()
        self.capital = 1000 # fixed capital in static   #constant        
        self.total_portfolio_static = 1000
        self.pnl = []
        self.total_portfolio_compound = 1000
        self.static_balance = []
        self.compound_balance = []
        #self.hist['compound_balance'] = self.compound_balance
        self.total_pnl = 0 #compounding
        self.total_pnl_cumm = []
        self.number_of_btc=0
        self.fee = 0.0015
        self.trades = 0
        self.win_trades = 0
        self.lose_trades = 0
        self.benchmark = 0
        self.win_rate = 0
        self.gross_profit = 0
        self.net_profit = 0
        self.avg_profit = 0
        self.max_drawdown = 0
        self.largest_win = 0
        self.avg_win = 0
        self.largest_loss = 0
        self.avg_loss = 0
        self.max_holding_period = 0
        self.avg_holding_period = 0
        self.max_dip = 0
        self.avg_dip = 0
        self.sharpe_ratio = 0
        self.sortino_ratio = 0
        self.is_closed = True
        self.initial_price = 0
        self.final_price = 0
        #extract to and from datetime
        self.from_datetime = self.hist['datetime'][0]
        self.to_datetime = self.hist['datetime'][len(self.hist)-1]
        self.entry_times = []
        self.exit_times = []
        self.dips = []
        self.multiple = False
        self.total_compound_fee = 0

    # 5 cases:-
    # 1. open long
    # 2. close long
    # 3. open short
    # 4. close short
    # 5. do nothing

    def extract_signals(self):
        #extract signals from strat
        signals = self.strat['signals']
        return signals
    
    def transaction_cost(self, price):
        return self.fee*price
    
    def open_long(self, index):
        #buy at opening price
        self.initial_price = self.hist['close'][index]
        self.is_closed = False
        self.trade_signal = 1
        self.entry_times.append(self.hist['datetime'][index])
        #static
        if self.is_static:
            self.total_portfolio_static -= self.transaction_cost(self.capital)
            self.static_balance.append(self.total_portfolio_static)
            self.total_portfolio_static -= self.capital
            self.number_of_btc += self.capital/self.hist['close'][index]
        #compounding
        else:
            self.number_of_btc += self.total_portfolio_compound/self.hist['close'][index]
            curr_transaction_cost = self.transaction_cost(self.total_portfolio_compound)
            self.total_compound_fee += curr_transaction_cost
            self.total_portfolio_compound = 0
            self.total_portfolio_compound -= curr_transaction_cost
    

    def open_short(self, index):
        #sell at closing price
        self.initial_price = self.hist['close'][index]
        self.is_closed = False
        self.entry_times.append(self.hist['datetime'][index])
        self.trade_signal = -1
        #static
        if self.is_static:
            self.number_of_btc -= self.capital/self.hist['close'][index]
            self.total_portfolio_static -= self.transaction_cost(self.capital)
            self.static_balance.append(self.total_portfolio_static)
            self.total_portfolio_static += self.capital
        #compounding
        else:
            self.number_of_btc -= self.total_portfolio_compound/self.hist['close'][index]
            self.total_portfolio_compound -= self.transaction_cost(self.total_portfolio_compound)
            self.total_compound_fee += self.transaction_cost(self.total_portfolio_compound)
            self.total_portfolio_compound += (-self.number_of_btc)*self.hist['close'][index]
             


    def close_long(self, index):
        #sell at closing price
        self.final_price = self.hist['close'][index]
        self.is_closed = True
        self.trades += 1
        current_pnl = self.number_of_btc*(self.final_price-self.initial_price)
        if self.is_static:
            current_pnl -= self.transaction_cost(self.capital)
        else:
            current_pnl -= self.transaction_cost(self.total_portfolio_compound)
        self.total_pnl += current_pnl
        self.total_pnl_cumm.append(self.total_pnl)
        self.pnl.append(current_pnl)
        self.trade_signal = 0
        self.exit_times.append(self.hist['datetime'][index])
        if current_pnl < 0 :
            dip = -((self.final_price - self.initial_price)/self.initial_price)
            self.dips.append(dip)
        #static
        if self.is_static:
            self.total_portfolio_static += self.number_of_btc*self.hist['close'][index]
            self.static_balance.append(self.total_portfolio_static)
            self.number_of_btc = 0
        #compounding
        else:
            self.total_portfolio_compound += self.number_of_btc*self.hist['close'][index]
            self.compound_balance.append(self.total_portfolio_compound) 
            self.number_of_btc = 0


    def close_short(self, index):
        #buy at open      
        self.final_price = self.hist['close'][index]
        self.is_closed = True
        self.trades += 1
        current_pnl = (-self.number_of_btc)*(self.initial_price-self.final_price)
        if self.is_static:
            current_pnl -= self.transaction_cost(self.capital)
        else:
            current_pnl -= self.transaction_cost(self.total_portfolio_compound)
        self.total_pnl += current_pnl
        self.total_pnl_cumm.append(self.total_pnl)
        self.pnl.append(current_pnl)
        self.trade_signal = 0
        self.exit_times.append(self.hist['datetime'][index])
        
        if current_pnl < 0 :
            dip = ((self.final_price - self.initial_price)/self.initial_price)
            self.dips.append(dip)
        #static
        if self.is_static:
            self.total_portfolio_static += (self.number_of_btc)*self.hist['close'][index]
            self.static_balance.append(self.total_portfolio_static)
            self.number_of_btc = 0
        #compounding
        else:
            self.total_portfolio_compound += (self.number_of_btc)*self.hist['close'][index]
            self.compound_balance.append(self.total_portfolio_compound)
            self.number_of_btc = 0 

    def do_nothing(self, index):
        #evil floating point bit hack
        self.static_balance.append(self.total_portfolio_static+self.number_of_btc*self.hist['close'][index])
        pass
    
    #plot graph of compound balance vs time 
    def plot_graph(self):
        # x axis is datetime
        # y axis is compound balance
        # plot compound balance vs time
        plt.plot(self.compound_balance)
        plt.xlabel('No of Trades')
        plt.ylabel('compound balance')
        plt.show()
   
    def calculate_net_profit(self):
        self.net_profit = sum(pnl for pnl in self.pnl)

    def calculate_max_drawdown(self):
    #calculate max_drawdown perc using static balance
        stat_balance = self.static_balance
        static_balance_series = pd.Series(stat_balance)
        max_balance = static_balance_series.cummax()
        drawdown = (static_balance_series - max_balance)
        max_drawdown_index = drawdown.idxmin()
        max_drawdown = (drawdown[max_drawdown_index]/max_balance[max_drawdown_index])*100
        self.max_drawdown = max_drawdown


        
    def calculate_holding_periods(self):
    # Assuming you have a list of timestamps representing entry and exit times
        entry_times = [datetime.strptime(time_str, '%Y-%m-%d %H:%M:%S') for time_str in self.entry_times]
        exit_times = [datetime.strptime(time_str, '%Y-%m-%d %H:%M:%S') for time_str in self.exit_times]

    # Calculate the holding period for each trade
        holding_periods = [(exit_time - entry_time).total_seconds() / 3600 for entry_time, exit_time in zip(entry_times, exit_times)]

        if holding_periods:
            self.max_holding_period = max(holding_periods)
            self.avg_holding_period = sum(holding_periods) / len(holding_periods)
        else:
            self.max_holding_period = 0
            self.avg_holding_period = 0

    def calculate_benchmark(self):
        self.benchmark = ((self.hist['close'][len(self.hist) - 1]-self.hist['close'][0])/self.hist['close'][0])*self.capital
    
    def calculate_ratios(self):
        #for sharpe ratio, find return of portfolio and subtract risk free rate, then divide by standard deviation of portfolio
        #for sortino ratio, find return of portfolio and subtract risk free rate, then divide by standard deviation of negative returns
        
    
        # Calculate Sharpe ratio
        returns = pd.DataFrame(self.pnl)
        avg_return = returns.mean()
        risk_free_rate = 0.02
        std_dev = returns.std()
        self.sharpe_ratio = (avg_return - risk_free_rate) / std_dev

        # Calculate Sortino ratio
        downside_returns = returns[returns < 0]
        avg_downside_return = downside_returns.mean()
        downside_std_dev = downside_returns.std()
        self.sortino_ratio = (avg_downside_return - risk_free_rate) / downside_std_dev
        
        self.sharpe_ratio = self.sharpe_ratio.tolist()[0]
        self.sortino_ratio = self.sortino_ratio.tolist()[0]


        
    def calculate_metrics_static(self):
        # Calculate static performance metrics
        self.total_trades_static = self.trades
        self.win_trades_static = sum(pnl > -(self.capital*self.fee) for pnl in self.pnl)
        self.lose_trades_static = self.total_trades_static - self.win_trades_static
        self.win_rate_static = self.win_trades_static / self.total_trades_static if self.total_trades_static > 0 else 0
        self.calculate_net_profit()
        self.gross_profit = self.net_profit + self.trades*self.capital*self.fee
        self.avg_profit_static = self.net_profit / (self.trades)
        self.largest_win = max(self.pnl)
        self.largest_loss = min(self.pnl)
        self.avg_win_static = (sum(pnl for pnl in self.pnl if pnl > 0)//self.win_trades_static) if self.win_trades_static > 0 else 0
        self.avg_loss_static = (sum(pnl for pnl in self.pnl if pnl < 0)//self.lose_trades_static) if self.lose_trades_static > 0 else 0
        self.calculate_max_drawdown()
        self.calculate_holding_periods()
        self.calculate_benchmark()
        self.calculate_ratios()
        self.max_dip = max(self.dips)
        self.avg_dip = sum(self.dips)/len(self.dips)

    def calculate_metrics_compound(self):
        # Calculate compound performance metrics
        self.initial_balance_compound = self.capital
        self.num_trades_compound = self.trades
        self.max_pnl_compound = max(self.pnl) 
        self.min_pnl_compound = min(self.pnl)
        self.peak_portfolio_balance_compound = max(self.compound_balance) if self.compound_balance else self.initial_balance_compound
        self.lowest_portfolio_balance_compound = min(self.compound_balance) 
        self.end_portfolio_balance_compound = self.compound_balance[-1] if self.compound_balance else self.initial_balance_compound
        

    
    def backtest(self, static,multiple):
        self.multiple = multiple
        if self.multiple == False:
            self.is_static = static
            for index, signal in enumerate(self.signals):
                if signal == 1 and self.is_closed:
                    self.open_long(index)
                elif signal == -1 and not self.is_closed:
                    self.close_long(index)
                elif signal == -1 and self.is_closed:
                    self.open_short(index)
                elif signal == 1 and not self.is_closed:
                    self.close_short(index)
                else:
                    self.do_nothing(index)
            
            if not self.is_static:
                self.plot_graph()
                self.calculate_metrics_compound()
            else:    
                self.calculate_metrics_static()
        else:
            self.is_static = static
            for index, signal in enumerate(self.signals):
                if signal == 1 :
                    if self.trade_signal == 1:
                        self.close_long(index)
                    elif self.trade_signal == -1: 
                        self.close_short(index)
                    self.open_long(index)
                elif signal == -1 :
                    if self.trade_signal == 1:
                        self.close_long(index)
                    elif self.trade_signal == -1: 
                        self.close_short(index)
                    self.open_short(index)
                else:
                    self.do_nothing(index)
                    
            if not self.is_static:
                self.plot_graph()
                self.calculate_metrics_compound()
            else:    
                self.calculate_metrics_static()
        

In [ ]:
df = pd.DataFrame(columns = ['ema1','ema2','ema3','atr window','atr multiple', 'returns' , 'drawdown'])
l = 0

for u in range(9 , 14):
    for o in range(44 , 48):
        for p in range(61,66):
            for a in range(11,16):
                for s in range(20,25):
                    final_df["ema_25"]=final_df["close"].ewm(span=u, adjust=False).mean()
                    final_df["ema_50"]=final_df["close"].ewm(span=o, adjust=False).mean()
                    final_df["ema_70"]=final_df["close"].ewm(span=p, adjust=False).mean()
                    print(l)
                    l+=1
                    atr = ta.atr(final_df['high'], final_df['low'], final_df['close'], timeperiod=a)
        
# Assign                      values to a new column in your DataFrame
                    final_df['atr'] = atr
                    
                    signals_final=[0,0]
                    temp=[]
                    curr_sig=0
                    closed=True
                    import statistics
                    from statistics import mode
                        #### set conditions for long_entry and long_exit
                    # print(len(final_df))
                    long_entry_price=None
                    short_entry_price=None
                    for i in range(2,len(final_df)):
                        if final_df.iloc[i].isna().any():
                            signals_final.append(0)
                    
                        else:
                            dic={}
                            long_entry=strat_long_entry(final_df,i)
                    
                            short_exit=strat_short_exit(final_df,i)
                            short_entry=strat_short_entry(final_df,i)
                            long_exit=strat_long_exit(final_df,i)
                            #stop loss ocnditions
                            con1=short_stop_loss(i,short_entry_price,s/10)
                            con2=long_stop_loss(i,long_entry_price,s/10)
                    
                    
                            if long_entry or short_exit or con1:
                                  #go long
                                  if long_entry and closed==True:
                                      #previous trade closed, can open new one
                                      signals_final.append(1)
                                      long_entry_price=final_df["close"].iloc[i]
                                      closed=False
                                      curr_sig=1
                                  else:
                                                           
                                      if ((con1 or short_exit)and curr_sig==-1):
                                          closed=True
                                          curr_sig=0
                                          signals_final.append(1)
                                          short_entry_price=None
                                      else:
                                          signals_final.append(0)
                              # set -1
                            elif short_entry or long_exit or con2:
                                  if short_entry and closed==True:
                                      signals_final.append(-1)
                                      short_entry_price=final_df["close"].iloc[i]
                                      closed=False
                                      curr_sig=-1
                                  else:
                                      
                                      if (long_exit or con2 ) and curr_sig==1:
                                          
                                          closed=True
                                          curr_sig=0
                                          signals_final.append(-1)
                                          long_entry_price=None
                                      else:
                                          signals_final.append(0)
                            else:
                                signals_final.append(0)
                    bt = backtest(final_df, pd.DataFrame(signals_final , columns = ['signals']))
                    bt.backtest(static=True,multiple = False)
                    df.loc[len(df)] = [u,o,p,a,s , bt.net_profit , bt.max_drawdown]
                    print(df.loc[len(df)-1])
            

In [ ]:
df.sort_values(by = 'returns')

In [ ]:
df.sort_values(by = 'drawdown')